# ERA V5 · Session 14 — a dense model grown into a mixture of experts

> Train a Linear model and convert that into an MoE! Your call on model size and data trained
> on, but must show they continue to train and reduce loss!

This notebook runs a short version of the real run live: train a dense byte-level transformer,
convert it into a mixture of experts four different ways, show the conversion can be **exact**,
and then keep training. The full-length numbers are in `README.md` and `results.json`; the
training log the assignment asks for is `run.log`.

A GPU helps. The conversion checks run anywhere.

In [1]:
import os, sys, json
if not os.path.exists('train.py'):              # on Colab: fetch the code
    !git clone -q https://github.com/sandeepkunkunuru/era-v5.git 2>/dev/null || true
    os.chdir('era-v5/session-14-moe')
sys.path.insert(0, os.getcwd())
import torch, train as T, model as M
print('torch', torch.__version__, '| device', T.DEV)

torch 2.12.1+cu130 | device cuda


## The two shapes

The mixture-of-experts model is built so that each token uses **exactly the dense model's
feed-forward width**. The dense inner width is 1,024; the experts are 128 wide and 8 are chosen,
and 8 x 128 = 1,024. So the only thing that changes is how many feed-forward parameters the layer
*stores*.

In [2]:
dense = M.GPT(M.Config())
moe = M.GPT(M.Config(moe=True))
d, m = dense.param_counts(), moe.param_counts()
print(f"dense  {d['total']:>12,} total  {d['active']:>12,} active")
print(f"moe    {m['total']:>12,} total  {m['active']:>12,} active")
print(f"ratio  {m['total']/d['total']:>12.2f}x {m['active']/d['active']:>12.2f}x")

dense    21,538,176 total    21,538,176 active
moe      64,152,960 total    21,685,632 active
ratio          2.98x         1.01x


## 1. Train the dense model

Short here — 1M tokens of Session 4's cleaned corpus, as bytes. The real run used 12M.

In [3]:
row = T.run(dense, 'nb-dense', 1_000_000, batch=8, accum=1)
dense_val = row['val_loss']
dense = dense.cpu()
print('dense validation loss', dense_val)

[nb-dense] step 0/244 loss 5.5586 lr 6.00e-06


[nb-dense] step 50/244 loss 3.2989 lr 1.41e-04


[nb-dense] step 100/244 loss 2.7755 lr 2.01e-04


[nb-dense] step 150/244 loss 2.6723 lr 1.16e-04


[nb-dense] step 200/244 loss 2.5652 lr 5.02e-05


[nb-dense] step 243/244 loss 2.4393 lr 3.00e-05


[nb-dense] done: val 2.5352 · 57,205 tok/s · 1,291 MiB peak


dense validation loss 2.5352


## 2. The conversion can be exact

Force the router onto the inherited experts and scale the down-projection by top-k. Then the
mixture-of-experts layer *is* the dense layer: `partition` and `clone` hold exactly the dense
network's neurons between them, and a weighted sum whose weights add to one, times k, puts it
back together. `drop` cannot, because half of each expert is new.

In [4]:
va = T.Batches('val', 8)
for meth in ('clone', 'partition', 'drop'):
    net = M.upcycle(dense, meth, M.Config(moe=True, balance='bias'), scale_down=True).to(T.DEV)
    for lay in net.moe_layers():
        torch.nn.init.zeros_(lay.router.weight); lay.bias.zero_(); lay.bias[:8] = 1.0
    v = T.evaluate(net, va)
    print(f'{meth:10s} val {v:.6f}   delta {v - dense_val:+.6f}')
    del net; torch.cuda.empty_cache() if T.DEV == 'cuda' else None

clone      val 2.535212   delta +0.000012


partition  val 2.535212   delta +0.000012


drop       val 2.833052   delta +0.297852


## 3. The scaling factor is not a detail

With a real router the k kept weights are each about 1/k, so the layer's output is about 1/k of
the dense layer's. The brief calls the correction a *routed scaling factor* (2.5 in DeepSeek-V3,
1.5 in V4.1-Flash) and notes the papers give the value without explaining it. Here the right
value is known exactly — it is top-k.

In [5]:
for meth in ('random', 'clone', 'partition', 'drop'):
    for sc in (False, True):
        net = M.upcycle(dense, meth, M.Config(moe=True), scale_down=sc).to(T.DEV)
        v = T.evaluate(net, va)
        print(f'{meth:10s} scale_down={str(sc):5s} val {v:7.4f}  delta {v - dense_val:+.4f}')
        del net; torch.cuda.empty_cache() if T.DEV == 'cuda' else None

random     scale_down=False val  3.5509  delta +1.0157


random     scale_down=True  val  3.5531  delta +1.0179


clone      scale_down=False val  3.4652  delta +0.9300


clone      scale_down=True  val  2.5649  delta +0.0297


partition  scale_down=False val  3.4675  delta +0.9323


partition  scale_down=True  val  2.5655  delta +0.0303


drop       scale_down=False val  3.5254  delta +0.9902


drop       scale_down=True  val  2.8597  delta +0.3245


## 4. Does it keep training, and does the loss drop?

The assignment's actual question. Each grown model gets 600k further tokens, against a dense
control given the same.

In [6]:
ctrl = M.GPT(M.Config()); ctrl.load_state_dict(dense.state_dict())
rows = [T.run(ctrl, 'nb-dense-control', 600_000, batch=8, accum=1)]
del ctrl; torch.cuda.empty_cache() if T.DEV == 'cuda' else None
for meth in ('random', 'partition'):
    net = M.upcycle(dense, meth, M.Config(moe=True, balance='bias'),
                    scale_down=(meth != 'random'))
    rows.append(T.run(net, f'nb-{meth}', 600_000, batch=8, accum=1))
    del net; torch.cuda.empty_cache() if T.DEV == 'cuda' else None
for r in rows:
    print(f"{r['tag']:18s} val {r['val_loss']:7.4f}  {r['tokens_per_s']:>7,} tok/s"
          f"  MaxVio {r.get('maxvio','--')}  dead {r.get('dead','--')}")

[nb-dense-control] step 0/146 loss 2.4855 lr 6.00e-06


[nb-dense-control] step 50/146 loss 2.5145 lr 1.18e-04


[nb-dense-control] step 100/146 loss 2.6057 lr 8.85e-05


[nb-dense-control] step 145/146 loss 2.5181 lr 3.00e-05


[nb-dense-control] done: val 2.498 · 58,017 tok/s · 1,310 MiB peak


...

[nb-partition] step 50/146 loss 2.5196 lr 1.18e-04


[nb-partition] step 100/146 loss 2.6237 lr 8.85e-05


[nb-partition] step 145/146 loss 2.5240 lr 3.00e-05


[nb-partition] done: val 2.5075 · 13,147 tok/s · 3,396 MiB peak · MaxVio 0.1445 · 0/384 dead


nb-dense-control   val  2.4980   58,017 tok/s  MaxVio --  dead --
nb-random          val  2.5958   13,716 tok/s  MaxVio 0.1791  dead 0
nb-partition       val  2.5075   13,147 tok/s  MaxVio 0.1445  dead 0


## 5. Balancing is what keeps experts alive

Same init, same tokens, balancing off against the auxiliary-loss-free bias. Watch the
dead-expert count, not the loss: an unused expert costs nothing today and costs its whole
capacity for the rest of training.

In [7]:
for bal in ('none', 'bias'):
    net = M.upcycle(dense, 'partition', M.Config(moe=True, balance=bal), scale_down=True)
    r = T.run(net, f'nb-balance-{bal}', 600_000, batch=8, accum=1)
    print(f"{bal:6s} val {r['val_loss']:.4f}  MaxVio {r['maxvio']}"
          f"  worst layer {r['maxvio_worst']}  dead {r['dead']}/{r['dead_of']}")
    del net; torch.cuda.empty_cache() if T.DEV == 'cuda' else None

[nb-balance-none] step 0/146 loss 2.5197 lr 6.00e-06


[nb-balance-none] step 50/146 loss 2.5206 lr 1.18e-04


[nb-balance-none] step 100/146 loss 2.6179 lr 8.85e-05


[nb-balance-none] step 145/146 loss 2.5240 lr 3.00e-05


[nb-balance-none] done: val 2.5033 · 13,422 tok/s · 3,396 MiB peak · MaxVio 1.9613 · 2/384 dead


...

[nb-balance-bias] step 50/146 loss 2.5215 lr 1.18e-04


[nb-balance-bias] step 100/146 loss 2.6196 lr 8.85e-05


[nb-balance-bias] step 145/146 loss 2.5338 lr 3.00e-05


[nb-balance-bias] done: val 2.5062 · 12,995 tok/s · 3,405 MiB peak · MaxVio 0.1564 · 0/384 dead


bias   val 2.5062  MaxVio 0.1564  worst layer 0.3236  dead 0/384


## 6. The full-length numbers

Read from `results.json`, which came from `python train.py`.

In [8]:
R = json.loads(open('results.json').read())
print('dense      ', {k: R['dense'][k] for k in ('tokens', 'val_loss', 'tokens_per_s')})
for r in R['grow']:
    print(f"{r['tag']:16s} val {r['val_loss']:7.4f}  MaxVio {r.get('maxvio','--')}"
          f"  dead {r.get('dead','--')}")
for r in R['balance']:
    print(f"balance {r['balance']:5s} val {r['val_loss']:7.4f}  MaxVio {r['maxvio']}"
          f"  dead {r['dead']}/{r['dead_of']}")

dense       {'tokens': 11993088, 'val_loss': 1.1881, 'tokens_per_s': 63240}
dense-control    val  0.9768  MaxVio --  dead --
grow:random      val  1.0377  MaxVio 0.0154  dead 0
grow:clone       val  0.9845  MaxVio 0.0139  dead 0
grow:partition   val  0.9910  MaxVio 0.0138  dead 0
grow:drop        val  0.9890  MaxVio 0.0149  dead 0
balance bias  val  0.9910  MaxVio 0.0138  dead 0/384
balance none  val  0.9836  MaxVio 2.1498  dead 61/384
balance aux   val  0.9844  MaxVio 1.8435  dead 0/384


## What to take away

- A dense model converts into a mixture of experts **without losing what it learned**, as long as
  the output is scaled for the router's weights. Without that scaling the conversion throws away
  about 0.8 nats on the spot.
- Inherited experts beat experts started from scratch by a wide margin at this budget. The
  `random` run is the same architecture with nothing carried over, and it is far behind.
- Balancing does not buy loss in the short run. It buys **capacity that stays usable**: with no
  balancing, experts go dead and never come back.
- On one GPU the mixture of experts is slower at equal active parameters, because 32 small matrix
  multiplications per layer replace one large one. That is a dispatch artifact, not the
  communication cost a real multi-GPU run pays — which this setup cannot show at all.